In [1]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
    --------------------------------------- 1.0/48.9 MB 4.0 MB/s eta 0:00:12
   - -------------------------------------- 1.6/48.9 MB 3.0 MB/s eta 0:00:16
   -- ------------------------------------- 2.6/48.9 MB 3.5 MB/s eta 0:00:14
   -- ------------------------------------- 3.4/48.9 MB 3.7 MB/s eta 0:00:13
   --- ------------------------------------ 4.5/48.9 MB 3.8 MB/s eta 0:00:12
   ---- ----------------------------------- 5.5/48.9 MB 3.9 MB/s eta 0:00:12
   ----- ---------------------------------- 6.6/48.9 MB 4.0 MB/s eta 0:00:11
   ------ --------------------------------- 7.6/48.9 MB 4.1 MB/s eta 0:00:10
   ------ --------------------------------- 8.4/48.9 MB 4.1 MB/s eta 0:00:10
   ------- -------------------------------- 9.4/48.9 MB 4.2 MB/s eta 0:00:10
   -------- ------------------------------- 10.5/48.9 MB 4.2 MB/s eta 0:00:10
   -------- 

In [2]:
from xgboost import XGBRegressor

print("XGBoost is ready!")

XGBoost is ready!


In [3]:
import pandas as pd
import numpy as np
import os
import time

from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    explained_variance_score
)

from xgboost import XGBRegressor

print("C9 imports completed successfully!")

C9 imports completed successfully!


In [4]:
# C9 Step 3 — Load Energy Dataset

energy_fe_path = "../../Feature_engineered_datasets/energy_feature_engineered.csv"

energy_df = pd.read_csv(energy_fe_path)

print("Energy dataset shape:", energy_df.shape)
print("\nColumns:")
print(energy_df.columns.tolist())

print("\nMissing values:")
print(energy_df.isnull().sum().sum())

print("\nTarget summary:")
print(energy_df["EnergyConsumption"].describe())

Energy dataset shape: (1000, 19)

Columns:
['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status', 'Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot']

Missing values:
0

Target summary:
count    1000.000000
mean       77.055873
std         8.144112
min        53.263278
25%        71.544690
50%        76.943696
75%        82.921742
max        99.201120
Name: EnergyConsumption, dtype: float64


In [5]:
# C9 Step 4 — Target and Leakage Control

target = "EnergyConsumption"

leakage_columns = [
    "Energy_Per_Occupant",
    "Renewable_Energy_Ratio",
    "Energy_Per_SquareFoot"
]

print("Target:", target)

print("\nLeakage columns:")
for col in leakage_columns:
    print(f"- {col}: {'Found' if col in energy_df.columns else 'Not Found'}")

# Same predictor set used in C7 and C8
energy_features_c9 = [
    "Temperature",
    "Humidity",
    "SquareFootage",
    "Occupancy",
    "HVACUsage",
    "LightingUsage",
    "RenewableEnergy",
    "DayOfWeek",
    "Holiday",
    "Hour",
    "Month",
    "Weekend",
    "HVAC_Status",
    "Lighting_Status"
]

X_c9 = energy_df[energy_features_c9].copy()
y_c9 = energy_df[target].copy()

print("\nX shape:", X_c9.shape)
print("y shape:", y_c9.shape)

print("\nMissing values in X:", X_c9.isnull().sum().sum())
print("Missing values in y:", y_c9.isnull().sum())

Target: EnergyConsumption

Leakage columns:
- Energy_Per_Occupant: Found
- Renewable_Energy_Ratio: Found
- Energy_Per_SquareFoot: Found

X shape: (1000, 14)
y shape: (1000,)

Missing values in X: 0
Missing values in y: 0


In [6]:
# C9 Step 5 — One-Hot Encoding

categorical_energy_columns_c9 = [
    "HVACUsage",
    "LightingUsage",
    "DayOfWeek",
    "Holiday",
    "Weekend"
]

X_c9_encoded = pd.get_dummies(
    X_c9,
    columns=categorical_energy_columns_c9,
    drop_first=True,
    dtype=int
)

print("Encoded feature shape:", X_c9_encoded.shape)

print("\nEncoded columns:")
print(X_c9_encoded.columns.tolist())

print("\nMissing values:", X_c9_encoded.isnull().sum().sum())

Encoded feature shape: (1000, 19)

Encoded columns:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'Hour', 'Month', 'HVAC_Status', 'Lighting_Status', 'HVACUsage_On', 'LightingUsage_On', 'DayOfWeek_Monday', 'DayOfWeek_Saturday', 'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday', 'DayOfWeek_Wednesday', 'Holiday_Yes', 'Weekend_Yes']

Missing values: 0


In [7]:
# C9 Step 6 — Train/Test Split

X_train_c9, X_test_c9, y_train_c9, y_test_c9 = train_test_split(
    X_c9_encoded,
    y_c9,
    test_size=0.20,
    random_state=42
)

print("Training data:")
print("X_train:", X_train_c9.shape)
print("y_train:", y_train_c9.shape)

print("\nTesting data:")
print("X_test:", X_test_c9.shape)
print("y_test:", y_test_c9.shape)

print("\nTraining target mean:", y_train_c9.mean())
print("Testing target mean:", y_test_c9.mean())

Training data:
X_train: (800, 19)
y_train: (800,)

Testing data:
X_test: (200, 19)
y_test: (200,)

Training target mean: 77.04935351116563
Testing target mean: 77.08195029880147


In [8]:
# C9 Step 7 — Random Forest OOF Predictions and Residuals

start_time_c9_oof = time.time()

kf_c9 = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_c9_oof = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Generate leakage-safe OOF predictions
y_train_c9_oof_pred = cross_val_predict(
    rf_c9_oof,
    X_train_c9,
    y_train_c9,
    cv=kf_c9,
    n_jobs=-1
)

# OOF residuals
y_train_c9_residual = (
    y_train_c9.values - y_train_c9_oof_pred
)

c9_oof_time = time.time() - start_time_c9_oof

print("C9 Random Forest OOF completed!")
print("-" * 45)
print(f"OOF predictions:       {len(y_train_c9_oof_pred)}")
print(f"Residuals:             {len(y_train_c9_residual)}")
print(f"Residual mean:         {np.mean(y_train_c9_residual):.6f}")
print(f"Residual std:          {np.std(y_train_c9_residual):.6f}")
print(f"OOF generation time:   {c9_oof_time:.4f} seconds")

C9 Random Forest OOF completed!
---------------------------------------------
OOF predictions:       800
Residuals:             800
Residual mean:         0.068406
Residual std:          5.443963
OOF generation time:   16.6677 seconds


In [9]:
# C9 Step 8 — XGBoost Residual Learner

start_time_c9_xgb = time.time()

xgb_c9 = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_c9.fit(
    X_train_c9,
    y_train_c9_residual
)

c9_xgb_time = time.time() - start_time_c9_xgb

print("C9 XGBoost residual model trained!")
print("-" * 45)
print(f"Residual samples:     {len(y_train_c9_residual)}")
print(f"XGBoost training time: {c9_xgb_time:.4f} seconds")

C9 XGBoost residual model trained!
---------------------------------------------
Residual samples:     800
XGBoost training time: 1.8670 seconds


In [10]:
# C9 Step 9 — Final RF + XGBoost Residual Prediction

start_time_c9_final = time.time()

# Train final Random Forest on the complete training data
rf_c9_final = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_c9_final.fit(
    X_train_c9,
    y_train_c9
)

# Base Random Forest prediction
y_pred_c9_rf = rf_c9_final.predict(X_test_c9)

# XGBoost residual correction
y_pred_c9_residual = xgb_c9.predict(X_test_c9)

# Final hybrid prediction
y_pred_c9_hybrid = (
    y_pred_c9_rf
    + y_pred_c9_residual
)

c9_final_time = time.time() - start_time_c9_final

# --------------------------------------------------
# Hybrid metrics
# --------------------------------------------------

c9_hybrid_mae = mean_absolute_error(
    y_test_c9,
    y_pred_c9_hybrid
)

c9_hybrid_mse = mean_squared_error(
    y_test_c9,
    y_pred_c9_hybrid
)

c9_hybrid_rmse = np.sqrt(c9_hybrid_mse)

c9_hybrid_r2 = r2_score(
    y_test_c9,
    y_pred_c9_hybrid
)

c9_hybrid_smape = (
    100
    * np.mean(
        2 * np.abs(y_test_c9 - y_pred_c9_hybrid)
        / (
            np.abs(y_test_c9)
            + np.abs(y_pred_c9_hybrid)
            + 1e-8
        )
    )
)

c9_hybrid_nrmse = (
    c9_hybrid_rmse / np.mean(y_test_c9)
) * 100

c9_hybrid_medae = median_absolute_error(
    y_test_c9,
    y_pred_c9_hybrid
)

c9_hybrid_ev = explained_variance_score(
    y_test_c9,
    y_pred_c9_hybrid
)

print("C9 Random Forest + XGBoost Residual Hybrid")
print("-" * 50)
print(f"MAE:                   {c9_hybrid_mae:.4f}")
print(f"MSE:                   {c9_hybrid_mse:.4f}")
print(f"RMSE:                  {c9_hybrid_rmse:.4f}")
print(f"R²:                    {c9_hybrid_r2:.4f}")
print(f"sMAPE:                 {c9_hybrid_smape:.4f}%")
print(f"NRMSE:                 {c9_hybrid_nrmse:.4f}%")
print(f"Median Absolute Error: {c9_hybrid_medae:.4f}")
print(f"Explained Variance:    {c9_hybrid_ev:.4f}")
print(f"Training Time:         {c9_final_time:.4f} seconds")

C9 Random Forest + XGBoost Residual Hybrid
--------------------------------------------------
MAE:                   4.6062
MSE:                   33.0237
RMSE:                  5.7466
R²:                    0.4958
sMAPE:                 6.0268%
NRMSE:                 7.4552%
Median Absolute Error: 3.9022
Explained Variance:    0.4971
Training Time:         1.9379 seconds


In [11]:
# C10 Step 1 — XGBoost Standalone Baseline

start_time_c10_xgb = time.time()

xgb_c10 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_c10.fit(
    X_train_c9,
    y_train_c9
)

y_pred_c10_xgb = xgb_c10.predict(X_test_c9)

c10_xgb_time = time.time() - start_time_c10_xgb

# Metrics
c10_xgb_mae = mean_absolute_error(
    y_test_c9,
    y_pred_c10_xgb
)

c10_xgb_mse = mean_squared_error(
    y_test_c9,
    y_pred_c10_xgb
)

c10_xgb_rmse = np.sqrt(c10_xgb_mse)

c10_xgb_r2 = r2_score(
    y_test_c9,
    y_pred_c10_xgb
)

c10_xgb_smape = (
    100
    * np.mean(
        2 * np.abs(y_test_c9 - y_pred_c10_xgb)
        / (
            np.abs(y_test_c9)
            + np.abs(y_pred_c10_xgb)
            + 1e-8
        )
    )
)

c10_xgb_nrmse = (
    c10_xgb_rmse / np.mean(y_test_c9)
) * 100

c10_xgb_medae = median_absolute_error(
    y_test_c9,
    y_pred_c10_xgb
)

c10_xgb_ev = explained_variance_score(
    y_test_c9,
    y_pred_c10_xgb
)

print("C10 XGBoost Standalone")
print("-" * 45)
print(f"MAE:                   {c10_xgb_mae:.4f}")
print(f"MSE:                   {c10_xgb_mse:.4f}")
print(f"RMSE:                  {c10_xgb_rmse:.4f}")
print(f"R²:                    {c10_xgb_r2:.4f}")
print(f"sMAPE:                 {c10_xgb_smape:.4f}%")
print(f"NRMSE:                 {c10_xgb_nrmse:.4f}%")
print(f"Median Absolute Error: {c10_xgb_medae:.4f}")
print(f"Explained Variance:    {c10_xgb_ev:.4f}")
print(f"Training Time:         {c10_xgb_time:.4f} seconds")

C10 XGBoost Standalone
---------------------------------------------
MAE:                   4.4234
MSE:                   30.1352
RMSE:                  5.4896
R²:                    0.5399
sMAPE:                 5.7896%
NRMSE:                 7.1217%
Median Absolute Error: 3.8316
Explained Variance:    0.5401
Training Time:         1.7332 seconds


In [12]:
# C11 Step 1 — Verify Existing Train/Test Data

print("C11 — XGBoost + MLP Stacking")
print("-" * 50)

print("X_train shape:", X_train_c9.shape)
print("X_test shape :", X_test_c9.shape)
print("y_train shape:", y_train_c9.shape)
print("y_test shape :", y_test_c9.shape)

print("\nTarget column: EnergyConsumption")
print("Train samples:", len(y_train_c9))
print("Test samples :", len(y_test_c9))

C11 — XGBoost + MLP Stacking
--------------------------------------------------
X_train shape: (800, 19)
X_test shape : (200, 19)
y_train shape: (800,)
y_test shape : (200,)

Target column: EnergyConsumption
Train samples: 800
Test samples : 200


In [13]:
# C11 Step 2 — Standardize Features for MLP

from sklearn.preprocessing import StandardScaler

scaler_c11 = StandardScaler()

X_train_c11_mlp = scaler_c11.fit_transform(X_train_c9)
X_test_c11_mlp = scaler_c11.transform(X_test_c9)

print("C11 MLP input preparation")
print("-" * 50)

print("Original training shape:", X_train_c9.shape)
print("Scaled training shape  :", X_train_c11_mlp.shape)

print("Original test shape    :", X_test_c9.shape)
print("Scaled test shape      :", X_test_c11_mlp.shape)

print("\nFirst 5 scaled training rows:")
print(X_train_c11_mlp[:5])

C11 MLP input preparation
--------------------------------------------------
Original training shape: (800, 19)
Scaled training shape  : (800, 19)
Original test shape    : (200, 19)
Scaled test shape      : (200, 19)

First 5 scaled training rows:
[[-0.60007123  0.11723233  0.51063898 -0.93096815  0.33478968 -0.93499464
  -0.58889842 -1.00752834  1.02532046 -1.00752834  1.02532046 -0.36927447
  -0.41388776 -0.41595736 -0.41181385  2.416114   -0.3822719  -0.90911166
  -0.64492573]
 [-1.01407377 -1.368855    1.71124264  0.81815003 -1.23448958 -0.64366233
  -0.58889842 -1.00752834 -0.97530483 -1.00752834 -0.97530483 -0.36927447
  -0.41388776 -0.41595736 -0.41181385 -0.41388776 -0.3822719   1.09997489
  -0.64492573]
 [ 0.41577638 -0.24227922  0.25573863 -0.93096815 -1.14691778  1.68699617
  -0.58889842 -1.00752834 -0.97530483 -1.00752834 -0.97530483  2.7080128
  -0.41388776 -0.41595736 -0.41181385 -0.41388776 -0.3822719  -0.90911166
  -0.64492573]
 [-0.49292495 -0.04673249 -1.28068469  0.4

In [14]:
# C11 Step 3 — Train MLP Regressor

from sklearn.neural_network import MLPRegressor
import time

start_time_c11_mlp = time.time()

mlp_c11 = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=30,
    random_state=42
)

mlp_c11.fit(
    X_train_c11_mlp,
    y_train_c9
)

c11_mlp_time = time.time() - start_time_c11_mlp

print("C11 MLP Training")
print("-" * 50)
print("Iterations:", mlp_c11.n_iter_)
print(f"Training Time: {c11_mlp_time:.4f} seconds")

C11 MLP Training
--------------------------------------------------
Iterations: 427
Training Time: 1.1956 seconds


In [15]:
# C11 Step 4 — MLP Standalone Evaluation

import numpy as np

# Generate predictions
y_pred_c11_mlp = mlp_c11.predict(
    X_test_c11_mlp
)

# Metrics
c11_mlp_mae = mean_absolute_error(
    y_test_c9,
    y_pred_c11_mlp
)

c11_mlp_mse = mean_squared_error(
    y_test_c9,
    y_pred_c11_mlp
)

c11_mlp_rmse = np.sqrt(
    c11_mlp_mse
)

c11_mlp_r2 = r2_score(
    y_test_c9,
    y_pred_c11_mlp
)

c11_mlp_smape = (
    100
    * np.mean(
        2 * np.abs(y_test_c9 - y_pred_c11_mlp)
        / (
            np.abs(y_test_c9)
            + np.abs(y_pred_c11_mlp)
            + 1e-8
        )
    )
)

c11_mlp_nrmse = (
    c11_mlp_rmse / np.mean(y_test_c9)
) * 100

c11_mlp_medae = median_absolute_error(
    y_test_c9,
    y_pred_c11_mlp
)

c11_mlp_ev = explained_variance_score(
    y_test_c9,
    y_pred_c11_mlp
)

print("C11 MLP Standalone")
print("-" * 50)
print(f"MAE:                   {c11_mlp_mae:.4f}")
print(f"MSE:                   {c11_mlp_mse:.4f}")
print(f"RMSE:                  {c11_mlp_rmse:.4f}")
print(f"R²:                    {c11_mlp_r2:.4f}")
print(f"sMAPE:                 {c11_mlp_smape:.4f}%")
print(f"NRMSE:                 {c11_mlp_nrmse:.4f}%")
print(f"Median Absolute Error: {c11_mlp_medae:.4f}")
print(f"Explained Variance:    {c11_mlp_ev:.4f}")
print(f"Training Time:         {c11_mlp_time:.4f} seconds")

C11 MLP Standalone
--------------------------------------------------
MAE:                   5.0702
MSE:                   40.1180
RMSE:                  6.3339
R²:                    0.3875
sMAPE:                 6.6291%
NRMSE:                 8.2171%
Median Absolute Error: 4.3416
Explained Variance:    0.3891
Training Time:         1.1956 seconds


In [16]:
# C11 Step 5 — Train XGBoost for Stacking

from xgboost import XGBRegressor
import time

start_time_c11_xgb = time.time()

xgb_c11 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_c11.fit(
    X_train_c9,
    y_train_c9
)

# Test predictions
y_pred_c11_xgb = xgb_c11.predict(
    X_test_c9
)

c11_xgb_time = time.time() - start_time_c11_xgb

print("C11 XGBoost component trained")
print("-" * 50)
print(f"Training Time: {c11_xgb_time:.4f} seconds")
print("Predictions:", len(y_pred_c11_xgb))

C11 XGBoost component trained
--------------------------------------------------
Training Time: 0.4508 seconds
Predictions: 200


In [17]:
# C11 Step 6 — Generate OOF Predictions for XGBoost + MLP

from sklearn.model_selection import KFold
from sklearn.base import clone
import numpy as np
import time

start_time_c11_oof = time.time()

kf_c11 = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Arrays for OOF predictions
oof_xgb_c11 = np.zeros(len(y_train_c9))
oof_mlp_c11 = np.zeros(len(y_train_c9))

for fold, (train_idx, val_idx) in enumerate(
    kf_c11.split(X_train_c9),
    start=1
):
    
    # -----------------------------
    # Split fold data
    # -----------------------------
    X_fold_train = X_train_c9.iloc[train_idx]
    X_fold_val = X_train_c9.iloc[val_idx]
    
    y_fold_train = y_train_c9.iloc[train_idx]
    y_fold_val = y_train_c9.iloc[val_idx]
    
    # -----------------------------
    # Scale MLP data using ONLY
    # fold training data
    # -----------------------------
    scaler_fold = StandardScaler()
    
    X_fold_train_mlp = scaler_fold.fit_transform(
        X_fold_train
    )
    
    X_fold_val_mlp = scaler_fold.transform(
        X_fold_val
    )
    
    # -----------------------------
    # XGBoost
    # -----------------------------
    xgb_fold = XGBRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
    
    xgb_fold.fit(
        X_fold_train,
        y_fold_train
    )
    
    oof_xgb_c11[val_idx] = xgb_fold.predict(
        X_fold_val
    )
    
    # -----------------------------
    # MLP
    # -----------------------------
    mlp_fold = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.2,
        n_iter_no_change=30,
        random_state=42
    )
    
    mlp_fold.fit(
        X_fold_train_mlp,
        y_fold_train
    )
    
    oof_mlp_c11[val_idx] = mlp_fold.predict(
        X_fold_val_mlp
    )
    
    print(f"Fold {fold} completed")

c11_oof_time = time.time() - start_time_c11_oof

print("\nC11 OOF Generation Complete")
print("-" * 50)
print("XGBoost OOF predictions:", len(oof_xgb_c11))
print("MLP OOF predictions    :", len(oof_mlp_c11))
print(f"OOF Generation Time    : {c11_oof_time:.4f} seconds")

Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed

C11 OOF Generation Complete
--------------------------------------------------
XGBoost OOF predictions: 800
MLP OOF predictions    : 800
OOF Generation Time    : 8.3005 seconds


In [18]:
# C11 Step 7 — Train Stacking Meta-Learner

from sklearn.linear_model import LinearRegression

# Create meta-training dataset
meta_X_c11 = np.column_stack([
    oof_xgb_c11,
    oof_mlp_c11
])

meta_y_c11 = y_train_c9.values

print("C11 Meta-Learner Dataset")
print("-" * 50)
print("Meta-feature shape:", meta_X_c11.shape)
print("Meta-target shape :", meta_y_c11.shape)

# Train meta-learner
meta_model_c11 = LinearRegression()

meta_model_c11.fit(
    meta_X_c11,
    meta_y_c11
)

print("\nMeta-learner trained successfully")

print("\nMeta-learner coefficients:")
print(f"XGBoost coefficient: {meta_model_c11.coef_[0]:.6f}")
print(f"MLP coefficient    : {meta_model_c11.coef_[1]:.6f}")
print(f"Intercept           : {meta_model_c11.intercept_:.6f}")

C11 Meta-Learner Dataset
--------------------------------------------------
Meta-feature shape: (800, 2)
Meta-target shape : (800,)

Meta-learner trained successfully

Meta-learner coefficients:
XGBoost coefficient: 0.906712
MLP coefficient    : 0.084686
Intercept           : 0.708556


In [19]:
# C11 Step 8 — Generate Final Stacking Predictions

start_time_c11_final = time.time()

# XGBoost test prediction
y_pred_c11_xgb_final = xgb_c11.predict(
    X_test_c9
)

# MLP test prediction
y_pred_c11_mlp_final = mlp_c11.predict(
    X_test_c11_mlp
)

# Create meta-level test features
meta_X_test_c11 = np.column_stack([
    y_pred_c11_xgb_final,
    y_pred_c11_mlp_final
])

# Final stacking prediction
y_pred_c11_stack = meta_model_c11.predict(
    meta_X_test_c11
)

c11_final_time = time.time() - start_time_c11_final

print("C11 Final Stacking Prediction")
print("-" * 50)

print("XGBoost predictions:", len(y_pred_c11_xgb_final))
print("MLP predictions    :", len(y_pred_c11_mlp_final))
print("Final predictions  :", len(y_pred_c11_stack))

print(f"\nPrediction Time: {c11_final_time:.4f} seconds")

print("\nFirst 5 final predictions:")
print(y_pred_c11_stack[:5])

C11 Final Stacking Prediction
--------------------------------------------------
XGBoost predictions: 200
MLP predictions    : 200
Final predictions  : 200

Prediction Time: 0.0073 seconds

First 5 final predictions:
[82.30392792 80.90002283 75.36711847 88.35948867 74.68664621]


In [20]:
# C11 Step 9 — Final Evaluation

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    explained_variance_score
)
import numpy as np

# Basic metrics
c11_mae = mean_absolute_error(
    y_test_c9,
    y_pred_c11_stack
)

c11_mse = mean_squared_error(
    y_test_c9,
    y_pred_c11_stack
)

c11_rmse = np.sqrt(c11_mse)

c11_r2 = r2_score(
    y_test_c9,
    y_pred_c11_stack
)

# sMAPE
c11_smape = np.mean(
    2 * np.abs(y_test_c9.values - y_pred_c11_stack) /
    (
        np.abs(y_test_c9.values) +
        np.abs(y_pred_c11_stack) +
        1e-8
    )
) * 100

# NRMSE
c11_nrmse = (
    c11_rmse /
    (y_test_c9.max() - y_test_c9.min())
) * 100

# Median Absolute Error
c11_median_ae = median_absolute_error(
    y_test_c9,
    y_pred_c11_stack
)

# Explained Variance
c11_ev = explained_variance_score(
    y_test_c9,
    y_pred_c11_stack
)

print("C11 — XGBoost + MLP Stacking Results")
print("=" * 60)

print(f"MAE                    : {c11_mae:.4f}")
print(f"MSE                    : {c11_mse:.4f}")
print(f"RMSE                   : {c11_rmse:.4f}")
print(f"R²                     : {c11_r2:.4f}")
print(f"sMAPE                  : {c11_smape:.4f}%")
print(f"NRMSE                  : {c11_nrmse:.4f}%")
print(f"Median Absolute Error  : {c11_median_ae:.4f}")
print(f"Explained Variance     : {c11_ev:.4f}")
print(f"Prediction Time        : {c11_final_time:.4f} seconds")

C11 — XGBoost + MLP Stacking Results
MAE                    : 4.4102
MSE                    : 30.0463
RMSE                   : 5.4814
R²                     : 0.5413
sMAPE                  : 5.7709%
NRMSE                  : 13.4796%
Median Absolute Error  : 3.7962
Explained Variance     : 0.5418
Prediction Time        : 0.0073 seconds


In [21]:
# Step 10 — Linear Regression Residual Analysis

from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Train LR using the existing training data
lr_diag = LinearRegression()

lr_diag.fit(
    X_train_c9,
    y_train_c9
)

# Predictions
y_pred_lr_diag = lr_diag.predict(X_test_c9)

# Residuals
lr_residuals = y_test_c9.values - y_pred_lr_diag

print("Linear Regression Residual Analysis")
print("=" * 55)

print(f"Residual Mean : {lr_residuals.mean():.4f}")
print(f"Residual Std  : {lr_residuals.std():.4f}")
print(f"Minimum       : {lr_residuals.min():.4f}")
print(f"Maximum       : {lr_residuals.max():.4f}")

print("\nLargest absolute residuals:")

largest_errors = np.argsort(
    np.abs(lr_residuals)
)[-10:][::-1]

for i in largest_errors:
    print(
        f"Actual={y_test_c9.iloc[i]:.4f}, "
        f"Predicted={y_pred_lr_diag[i]:.4f}, "
        f"Residual={lr_residuals[i]:.4f}"
    )

Linear Regression Residual Analysis
Residual Mean : 0.0924
Residual Std  : 5.1633
Minimum       : -17.9836
Maximum       : 12.0796

Largest absolute residuals:
Actual=61.4914, Predicted=79.4750, Residual=-17.9836
Actual=60.1442, Predicted=75.9775, Residual=-15.8332
Actual=72.8053, Predicted=85.4474, Residual=-12.6422
Actual=80.0186, Predicted=67.9389, Residual=12.0796
Actual=93.6175, Predicted=81.9200, Residual=11.6975
Actual=66.3371, Predicted=77.7687, Residual=-11.4315
Actual=76.0674, Predicted=87.3030, Residual=-11.2356
Actual=76.3996, Predicted=87.4395, Residual=-11.0400
Actual=66.0018, Predicted=76.1366, Residual=-10.1348
Actual=98.7618, Predicted=88.9565, Residual=9.8052


In [22]:
# Residual correlation with input features

residual_df = X_test_c9.copy()

residual_df["LR_Residual"] = lr_residuals

residual_correlations = (
    residual_df.corr(numeric_only=True)["LR_Residual"]
    .drop("LR_Residual")
    .sort_values(key=np.abs, ascending=False)
)

print("\nFeature–Residual Correlations")
print("=" * 55)
print(residual_correlations)


Feature–Residual Correlations
RenewableEnergy       -0.182414
DayOfWeek_Saturday    -0.167309
Weekend_Yes           -0.136846
DayOfWeek_Wednesday    0.135869
DayOfWeek_Monday      -0.104704
Occupancy              0.080169
Holiday_Yes           -0.066079
HVACUsage_On           0.051624
HVAC_Status            0.051624
Month                 -0.049315
Hour                   0.044117
Humidity              -0.042149
DayOfWeek_Thursday     0.039708
DayOfWeek_Tuesday      0.036819
SquareFootage          0.023126
Temperature           -0.022627
Lighting_Status        0.021642
LightingUsage_On       0.021642
DayOfWeek_Sunday      -0.018283
Name: LR_Residual, dtype: float64


In [24]:
# Step 11 — Temporal Structure Analysis

energy_time = pd.read_csv(
    "../../Feature_engineered_datasets/energy_feature_engineered.csv"
)

energy_time["Timestamp"] = pd.to_datetime(
    energy_time["Timestamp"]
)

energy_time = energy_time.sort_values(
    "Timestamp"
).reset_index(drop=True)

print("Temporal Structure Analysis")
print("=" * 55)

print("Number of records:", len(energy_time))

print("\nTimestamp range:")
print("Start:", energy_time["Timestamp"].min())
print("End  :", energy_time["Timestamp"].max())

print("\nFirst 10 timestamps:")
print(energy_time["Timestamp"].head(10).to_string(index=False))

print("\nLast 10 timestamps:")
print(energy_time["Timestamp"].tail(10).to_string(index=False))

# Time differences
time_diff = energy_time["Timestamp"].diff().dropna()

print("\nMost common time intervals:")
print(time_diff.value_counts().head(10))

# Target autocorrelation
energy_target = energy_time["EnergyConsumption"]

print("\nEnergyConsumption autocorrelation:")
for lag in [1, 2, 3, 6, 12, 24]:
    print(
        f"Lag {lag:2d}: "
        f"{energy_target.autocorr(lag=lag):.4f}"
    )

Temporal Structure Analysis
Number of records: 1000

Timestamp range:
Start: 2022-01-01 00:00:00
End  : 2022-02-11 15:00:00

First 10 timestamps:
2022-01-01 00:00:00
2022-01-01 01:00:00
2022-01-01 02:00:00
2022-01-01 03:00:00
2022-01-01 04:00:00
2022-01-01 05:00:00
2022-01-01 06:00:00
2022-01-01 07:00:00
2022-01-01 08:00:00
2022-01-01 09:00:00

Last 10 timestamps:
2022-02-11 06:00:00
2022-02-11 07:00:00
2022-02-11 08:00:00
2022-02-11 09:00:00
2022-02-11 10:00:00
2022-02-11 11:00:00
2022-02-11 12:00:00
2022-02-11 13:00:00
2022-02-11 14:00:00
2022-02-11 15:00:00

Most common time intervals:
Timestamp
0 days 01:00:00    999
Name: count, dtype: int64

EnergyConsumption autocorrelation:
Lag  1: -0.0003
Lag  2: -0.0311
Lag  3: -0.0433
Lag  6: 0.0146
Lag 12: 0.0600
Lag 24: 0.0084
